# Build Your Own Transformer

The exercises in this notebook walk you through building your own transformer. The weight matrices are supplied for you, and have been trained on Plato's dialogues. At the end, you will have a transformer that babbles fake Plato.

The transformer you will build is smaller than a commercial LLM (about 10% the size of GPT-2), so that it can run without special hardware. So its attempts at Plato won't be very good.

However, the same code could generate perfectly grammatical text on a more powerful computer, with just a few changes to the numbers at the top of the file (and corresponding weights).

## Exercise 1

This notebook needs two other files from the project: `weights.npz` and `plato.txt`. Make sure they're in the same folder as this notebook, then look over the rest of the notebook.

- The model's vocabulary is the 77 characters that appear in `plato.txt`: the English alphabet, most digits, spaces, line breaks, and a few punctuation symbols.
- The embedding space has 384 dimensions. There are 6 "blocks", in other words 6 attention layers each followed by an MLP layer. Each attention layer has 6 heads.
- The `generate()` function takes a prompt as input and uses the `transformer()` function to output 500 new tokens.
- Under each function you'll fill in, there's a cell that tests it. The expected output is shown in a comment, so you can check your work as you go.

Run the first two code cells, then the last one. For now it just repeats the same letter 500 times. (The test cells in between won't work until you fill in the functions they test.)

In [7]:
import numpy as np

n_input = 256
n_embed = 384
n_head = 6
n_block = 6

with open('plato.txt', 'r', encoding='utf-8') as f:
    text = f.read()

vocabulary = sorted(list(set(text)))

weights = np.load("weights.npz")

def generate(prompt):
    prompt = prompt.rjust(n_input)[-n_input:]    # make it 256 characters
    int_to_char = { i:ch for i,ch in enumerate(vocabulary) }
    for _ in range(500):
        new_token = transformer(prompt)
        new_char = int_to_char[new_token]
        print(new_char, end="", flush=True)
        prompt = prompt[1:] + new_char
    return None

def layer_norm(X, i, n):
    if i is not None:
        W = weights[f"blocks.{i}.ln{n}.weight"]
        b = weights[f"blocks.{i}.ln{n}.bias"]
    else:
        W = weights[f"ln_f.weight"]
        b = weights[f"ln_f.bias"]
    mean = np.mean(X, axis=0, keepdims=True)
    var = np.var(X, axis=0, keepdims=True)
    X_hat = (X - mean) / np.sqrt(var + 1e-5)
    return W[:, None] * X_hat + b[:, None]

## Exercise 2

Fill in the `transformer()` function. Following the flow of the transformer diagram in the chapter (the "Corrections" section), the steps are:

- Call `tokenize()` on `prompt` and store the result as `tokens`.
- Call `embed()` on `tokens` and store the result as `X`.
- Create a `for` loop with `n_block` iterations. Each iteration `i` should do the following steps:
    - Call `attention(X, i)`, add the result to `X`, and store that sum as the new value of `X`.
    - Call `mlp(X, i)`, add the result to `X`, and store that sum as the new value of `X`.
- Finally, call `unembed()` on `X`, and return the result.

In [8]:
def transformer(prompt):
    return 42

## Exercise 3

The `tokenize()` function needs to return a list of integers, one for each character in the `prompt`. For example, if the `prompt` is `Hello`, then `tokenize()` should return `[30, 55, 62, 62, 65]` because `H` is the 30th character in the vocabulary, `e` is 55th, etc.

Complete the `tokenize()` function so that it does this. Some tips:

> We use square brackets because `char_to_int` isn't a function, it's a different sort of Python object known as a `dict` (dictionary).

- I've provided `char_to_int` so you can find the integer corresponding to a given character. For example, `char_to_int['H']` returns `30`, and `char_to_int['e']` returns `55`. Note the square brackets here instead of parentheses.
- You can create an empty list called `foo` with `foo = []`.
- You can loop through the characters in a string by writing: `for c in my_string:`.
- You can append an element to the end of a list by writing: `foo.append(element)`.

In [ ]:
def tokenize(prompt):
    char_to_int = { ch:i for i,ch in enumerate(vocabulary) }
    # your code here
    return tokens

In [ ]:
tokenize("Hello")
# [30, 55, 62, 62, 65]

## Exercise 4

Next, the `embed()` function should take a list of integers and return a matrix. Each column of that matrix should be the sum of the token embedding and position embedding for the corresponding integer in the input.

The easiest way to do this is to make two matrices and then add them together.

- The columns of the first matrix should have the token embeddings.
- The columns of the second matrix should have the position embeddings.

I've provided two matrices to help with this:

- `tok_emb` is a `384 x 77` matrix. Column `i` is the token embedding for the `i`th character in the vocabulary.
- `pos_emb` is a `384 x 256` matrix. Column `i` is the position embedding for the `i`th position in the prompt.

Tip: you can select the columns you want from a matrix using a list of indices, like this:

```python
matrix = np.array([[1, 2, 3],
                   [4, 5, 6]])
indices = [0, 2, 2, 1]
selection = matrix[:, indices]
print(selection)
# [[1 3 3 2]
#  [4 6 6 5]]
```

In [ ]:
def embed(tokens):
    tok_emb = weights["token_embedding_table.weight"]
    pos_emb = weights["position_embedding_table.weight"]
    
    # your code here
    
    return X

In [ ]:
embed([1, 2, 3, 4, 5, 6, 7, 8] * 32)
# array([[ 0.07677806, -0.01879471, -0.04412892, ...
#        [-0.15232947,  0.0371886 , -0.01737451, ...
#        [-0.0167386 ,  0.02757457, -0.00623887, ...
#        ...

## Exercise 5

Fill in the `softmax()` function. This function should take a vector `x` as input and return a vector of probabilities.

Tips:

- The function `np.exp()` calculates $e^x$. You can pass it a vector and it will calculate $e^x$ for each element.
- The function `np.sum()` adds together a vector's elements.

In [ ]:
def softmax(x):
    # your code here
    return p

In [ ]:
softmax(np.array([-10, 0, 300]))
# array([2.33727929e-135, 5.14820022e-131, 1.00000000e+000])

## Exercise 6

This exercise introduces a technique that will greatly simplify the work below. In several places, we will need to perform the same linear operation on each column of some matrix `X`. That is, we'll have a matrix `W` and vector `b`, and we will need to calculate `W @ x + b`, for each column `x` of the matrix `X`. We could do this with a `for` loop, processing the columns of `X` one at a time. But there's a much simpler way.

**(a)** If we multiply `W @ X`, what will column `i` contain?

**(b)** Mathematically, you can't add a matrix and a vector. For example, the following operation is not defined:

$$
\begin{bmatrix}
  1 & 2\\
  3 & 4
\end{bmatrix} +
\begin{bmatrix}
  5\\
  6
\end{bmatrix}
$$

But NumPy won't give an error if you ask it to do this. Try it out and examine the result, writing the vector as a column: `np.array([[5], [6]])`. What does NumPy do to make this operation work? (NumPy's approach here is called *broadcasting*.)


**(c)** Based on the above, what's a simple way to calculate `W @ x + b` for each column `x` of `X`, without having to loop over the columns? (Assume `b` is a column, like the vector in part **(b)**. That's how the `b` vectors in `weights.npz` are stored.) Hint: you only need one line of very simple code (in fact it's just five characters of code, not counting whitespace).

In [ ]:
# try it out here

## Exercise 7

Implement the attention heads by filling in the `head()` function. This is the trickiest part, so let's step through it in three parts.

**(a)** Each column of `X` is an embedding vector. Multiply each embedding by `W_Q` to get its query vector, and store these query vectors as the columns of a new matrix `Q`.

Likewise, use `W_K` and `W_V` to create matrices `K` and `V`, whose columns are (respectively) the key and value vectors.

**(b)** Create a matrix called `scores`. The value at row `i`, column `j` should be the dot-product of column `i` of `Q` and column `j` of `K`, divided by `scale`. (Hint: transpose `Q`.)

I've provided some code to handle the remaining processing that needs to be done on `scores`, since this part is kind of fiddly. (First, this code makes sure each token only averages information from previous ones, since the prediction of the next token should only draw on preceding tokens. Second, it applies `softmax()` to convert `scores` to proper weights, storing them in the matrix `A`.)

**(c)** Each row of `A` contains a sequence of weights. Create a matrix where column `i` is a weighted average of the columns of `V`, using the weights from row `i` of `A`. Return that matrix.

In [ ]:
def head(X, i, h):
    scale = (n_embed // n_head) ** 0.5
    W_Q = weights[f"blocks.{i}.sa.heads.{h}.query.weight"]
    W_K = weights[f"blocks.{i}.sa.heads.{h}.key.weight"]
    W_V = weights[f"blocks.{i}.sa.heads.{h}.value.weight"]

    # your code here

    mask = np.triu(np.ones((n_input, n_input)), k=1).astype(bool)
    scores[mask] = -np.inf
    A = np.array([softmax(row) for row in scores])

    # your code here
    
    return X

In [ ]:
np.random.seed(138)
head(np.random.randn(n_embed, n_input), 2, 3)
# array([[ 0.99839488,  0.99859071,  0.98565628, ...
#        [-0.73916343, -0.7358345 , -0.52018611, ...
#        [-1.69293853, -1.69297562,  0.54085634, ...
#        ...

## Exercise 8

Fill in the `attention()` function. There are two parts here.

**(a)** Pass the input `X` through each attention head, then concatenate the results. You can do this with a `for` loop that runs `n_head` times. Each time, call `head(X, i, h)`, where `i` is the current attention layer and `h` is the number of the attention head. Be sure to store the output from each head as it's processed. When you have all the outputs stored in a list called `head_outputs`, you can concatenate them together with `np.concatenate(head_outputs, axis=0)`.

**(b)** Apply the linear projection $W \mathbf{x} + \mathbf{b}$ to each column of the result from step (a), and return the result.

In [ ]:
def attention(X, i):
    X = layer_norm(X, i, 1)

    # your code here

    W = weights[f"blocks.{i}.sa.proj.weight"]
    b = weights[f"blocks.{i}.sa.proj.bias"]
    
    # your code here

    return X

In [ ]:
np.random.seed(138)
attention(np.random.randn(n_embed, n_input), 4)
# array([[ 2.46088353, -0.62084314, -1.08064634, ...
#        [-0.11310164, -1.79105783, -1.11511373, ...
#        [-1.43946952, -1.48981745, -0.63142569, ...
#        ...

## Exercise 9

Next we'll fill in the `relu()` function. This function takes the role the logistic function played in the previous chapter.

In the previous chapter, the hidden layer of our our penguin-classification network applied the logistic function to each intermediate value before passing it on to the final, output layer. But here we'll use a different operation known as *ReLU* (short for *Rectified Linear Unit*).

> The ReLU function is often used in place of the logistic function in modern neural networks. It has a broadly similar shape, but is simpler and tends to work better in training.

The chapter's ReLU figure shows a graph of the ReLU function. The definition is:

$$
\text{ReLU}(x) =
  \begin{cases}
    0 & \text{ if } x \leq 0, \\
    x & \text{ if } x > 0.
  \end{cases}
$$

Fill in `relu()` so that it computes this function. It should take a matrix as input and apply ReLU to each entry. Tip: the function `np.maximum(a, X)` takes a number `a` and matrix `X` and returns the same matrix but with entries smaller than `a` replaced by `a`.

In [ ]:
def relu(X):
    # your code here
    return X

In [ ]:
A = np.array([[-100, 3], [57, -999]])
relu(A)
# array([[ 0,  3],
#        [57,  0]])

## Exercise 10

Fill in the `mlp()` function. The chapter's MLP diagram shows the basic structure of an MLP layer. Mathematically, the sequence of operations this network performs on an input vector $\mathbf{x}$ is:

$$
\begin{aligned}
\mathbf{y} &= \text{ReLU}(W_1 \mathbf{x} + \mathbf{b}_1) \\
\mathbf{z} &= W_2 \mathbf{y} + \mathbf{b}_2
\end{aligned}
$$

The `mlp()` function should take a matrix `X` as input, pass its columns through this sequence of operations, and return a matrix whose columns are the vectors that come out the other end.

In [ ]:
def mlp(X, i):
    X = layer_norm(X, i, 2)
    
    W1 = weights[f"blocks.{i}.ffwd.net.0.weight"]
    b1 = weights[f"blocks.{i}.ffwd.net.0.bias"]
    W2 = weights[f"blocks.{i}.ffwd.net.2.weight"]
    b2 = weights[f"blocks.{i}.ffwd.net.2.bias"]
    
    # your code here
    
    return X

In [ ]:
np.random.seed(138)
mlp(np.random.randn(n_embed, n_input), 5)
# array([[ 8.27854465e-01, -1.75778904e+00, -4.58605797e+00, ...
#        [-3.00895815e+00, -3.52312917e+00, -3.41911986e+00, ...
#        [ 1.72272443e-02, -3.54643906e+00, -8.98170212e+00, ...
#        ...

## Exercise 11

Fill in the `unembed()` function. This function takes a matrix `X` as input, and it returns an integer: the new token to be output. There are three basic steps to this process:

**(a)** Compare the embedding of each token in the vocabulary to the last column of `X`, by taking dot-products.

**(b)** Convert these dot-products to probabilities using `softmax()`.

**(c)** Choose the next token according to those probabilities.

Tips:

- You can get the last column of `X` with `X[:, -1]`.
- If `probs` is a vector of `n` probabilities, then

  ```python
  np.random.choice(len(probs), p=probs)
  ```

  will choose a number between `0` and `n` (excluding `n`) according to those probabilities.

In [ ]:
def unembed(X):
    X = layer_norm(X, None, None)
    
    W = weights["token_embedding_table.weight"]

    # your code here

    return new_token

In [ ]:
np.random.seed(138)
unembed(np.random.randn(n_embed, n_input))
# 55

You should now have a working transformer. Test it by running the last cell: its output should start the same way as the comment there. Then try running the `generate()` function on a prompt of your choice. (Your prompt can only use characters in the model's vocabulary. To see them, run `print(vocabulary)`.)

In [9]:
np.random.seed(138)    # delete this line to get different output
generate("SOCRATES:\n Tell me, Meno, can virtue be taught? Or is it acquired by practice?")
# MENO:
# Yes, Euthydemus, I do.
#
# SOCRATES:
# And is this wise, like fathers or brothers? ...

TTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTTT